<a href="https://colab.research.google.com/github/alwhshalkasr267-design/FinalProject/blob/main/finalproject_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import Required Libraries

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import zipfile
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [3]:
zip_path = '/content/drive/MyDrive/Obstacles_dataset.zip'
extracted_path = '/content/dataset/Obstacles_dataset'

if not os.path.exists(extracted_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_path)
    print("تم فك ضغط البيانات بنجاح في المسار المحلي.")
else:
    print("المجلد موجود مسبقاً.")

المجلد موجود مسبقاً.


Load Data

In [4]:
folder_path = '/content/dataset/Obstacles_dataset'

if len(os.listdir(folder_path)) == 1:
    folder_path = os.path.join(folder_path, os.listdir(folder_path)[0])

classes_names = sorted(os.listdir(folder_path))
print("Dataset Classes: ", classes_names)

Dataset Classes:  ['chair', 'door', 'fence', 'garbage_bin', 'obstacle', 'plant', 'pothole', 'stairs', 'table', 'vehicle']


In [5]:
# حساب عدد الصور في كل كلاس
i = 1
for folder in sorted(os.listdir(folder_path)):
    full_folder_path = os.path.join(folder_path, folder)
    if os.path.isdir(full_folder_path):
        num_images = len(os.listdir(full_folder_path))
        print(f"{i} Number of [{folder}] Class : {num_images} images")
        i += 1

1 Number of [chair] Class : 407 images
2 Number of [door] Class : 242 images
3 Number of [fence] Class : 179 images
4 Number of [garbage_bin] Class : 175 images
5 Number of [obstacle] Class : 423 images
6 Number of [plant] Class : 139 images
7 Number of [pothole] Class : 706 images
8 Number of [stairs] Class : 504 images
9 Number of [table] Class : 185 images
10 Number of [vehicle] Class : 604 images


In [6]:
images = []
labels = []

for label_idx, classes_name in enumerate(classes_names):
    class_dir = os.path.join(folder_path, classes_name)
    if os.path.isdir(class_dir):
        for file in os.listdir(class_dir):
            img_path = os.path.join(class_dir, file)
            images.append(img_path)
            labels.append(str(label_idx))

In [7]:
df = pd.DataFrame({
    "image": images,
    "label": labels
})

# خلط البيانات
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total images collected: {len(df)}")
df.head()

Total images collected: 3564


,image,label
0,/content/dataset/Obstacles_dataset/Obstacles_d...,7
1,/content/dataset/Obstacles_dataset/Obstacles_d...,0
2,/content/dataset/Obstacles_dataset/Obstacles_d...,6
3,/content/dataset/Obstacles_dataset/Obstacles_d...,4
4,/content/dataset/Obstacles_dataset/Obstacles_d...,8


In [8]:
# تنظيف الصور طبعا الصور عندي نظيفة هشغل الكود فقط عند اخر نقطة
from PIL import Image
from tqdm import tqdm

bad = []

for p in tqdm(df["image"]):
  try:
    with Image.open(p) as im:
      im.load()
  except:
    bad.append(p)

print("Bad images : ", len(bad))

df = df[~df["image"].isin(bad)].reset_index(drop=True)
print("Remaining images : ", len(df) )

100%|██████████| 3564/3564 [00:10<00:00, 327.60it/s]

Bad images :  0
Remaining images :  3564


In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
    )

print("Train: ", len(train_df))
print("Test: ", len(test_df))

Train:  2851
Test:  713


In [10]:
train_gen = ImageDataGenerator(
    rescale=1./255,          # تحجيم قيم البكسلات بين 0 و 1
    rotation_range=20,       # تدوير الصور
    width_shift_range=0.2,   # إزاحة يميناً ويساراً
    height_shift_range=0.2,  # إزاحة لأعلى وأسفل
    shear_range=0.2,         # قص الزوايا (مهم جداً للرؤية الجانبية للعوائق) - مضاف لرفع الدقة
    horizontal_flip=True,    # قلب الصور أفقياً
    fill_mode='nearest'      # تعبئة الفراغات الناتجة عن الإزاحة بشكل ذكي
)
test_gen = ImageDataGenerator(rescale=1./255)

In [11]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_it = train_gen.flow_from_dataframe(
    train_df,
    x_col="image",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_it = test_gen.flow_from_dataframe(
    test_df,
    x_col="image",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 2851 validated image filenames belonging to 10 classes.
Found 713 validated image filenames belonging to 10 classes.


In [12]:
from sklearn.utils import class_weight
import numpy as np

# 1. جلب لستة الكلاسات لكل الصور في التدريب
train_classes = train_it.classes

# 2. حساب الأوزان تلقائياً بناءً على تفاوت الأعداد
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_classes),
    y=train_classes
)

# 3. تحويلها إلى قاموس يفهمه كيرس
class_weights_dict = dict(enumerate(class_weights))

print("الأوزان المحسوبة للكلاسات:\n", class_weights_dict)

الأوزان المحسوبة للكلاسات:
 {0: np.float64(0.8745398773006134), 1: np.float64(1.4695876288659795), 2: np.float64(1.9937062937062937), 3: np.float64(2.0364285714285715), 4: np.float64(0.843491124260355), 5: np.float64(2.5684684684684687), 6: np.float64(0.5046017699115044), 7: np.float64(0.7074441687344913), 8: np.float64(1.9263513513513513), 9: np.float64(0.5902691511387164)}


In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout


model = Sequential([
    Conv2D(16, (3, 3), activation='relu',padding='same', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(32, (3, 3), activation='relu' ,padding='same'),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu',padding='same'),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu',padding='same'),
    MaxPooling2D((2, 2)),

    Conv2D(256, (3, 3), activation='relu',padding='same'),
    MaxPooling2D((2, 2)),


    Flatten(),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(classes_names), activation='softmax')
])


model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 224, 224, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 112, 112, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 112, 112, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 56, 56, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 56, 56, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 28, 28, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 14, 14, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │     3,211,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,638,314 (13.88 MB)

 Trainable params: 3,638,314 (13.88 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
from tensorflow.keras.callbacks import EarlyStopping

# مراقبة خسارة التحقق وإيقاف التدريب فوراً إذا توقف النموذج عن التحسن
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

In [19]:
# 2. تحديد مسار الحفظ داخل الدرايف
checkpoint_path = "/content/drive/MyDrive/my_model_checkpoint.keras"

# 3. إنشاء وظيفة الحفظ التلقائي
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=False, # يحفظ النموذج كاملاً
    monitor='val_accuracy',
    save_best_only=True
)

In [20]:
history = model.fit(
    train_it,
    epochs=35,
    validation_data=test_it,
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint_callback] # دمجنا الوظيفتين هنا داخل القائمة
)

Epoch 1/35
90/90 ━━━━━━━━━━━━━━━━━━━━ 272s 3s/step - accuracy: 0.1172 - loss: 2.3058 - val_accuracy: 0.1585 - val_loss: 2.3003
Epoch 2/35
 9/90 ━━━━━━━━━━━━━━━━━━━━ 5:31 4s/step - accuracy: 0.1681 - loss: 2.2918

KeyboardInterrupt: 

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# 1. حساب التوقعات بناءً على المولد test_it المرتب (shuffle=False جاهز من الخلية 5)
preds = model.predict(test_it)
pred_labels = np.argmax(preds, axis=1)

# 2. جلب التسميات الحقيقية من المولد
true_labels = test_it.classes

# 3. طباعة تقرير التصنيف النصي (Classification Report)
print("Classification Report:\n")
# استخدام classes_names مباشرة لتسمية الكلاسات
print(classification_report(true_labels, pred_labels, target_names=classes_names))

# 4. حساب ورسم مصفوفة الارتباك بشكل مرئي (Heatmap)
cm = confusion_matrix(true_labels, pred_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes_names,
            yticklabels=classes_names)
plt.ylabel('True Labels (القيم الحقيقية)')
plt.xlabel('Predicted Labels (توقعات النموذج)')
plt.title('Confusion Matrix Heatmap')
plt.show()

In [ ]:
# استرجاع النموذج الذي توقف عنده التدريب
my_net2 = tf.keras.models.load_model("/content/drive/MyDrive/my_model_checkpoint.keras")

# الآن يمكنك عمل fit مجدداً وسيبدأ من حيث توقف!

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Test Accuracy")
plt.legend()
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# 1. تأكد أن التوقعات مأخوذة بشكل صحيح
preds = model.predict(test_it)

# 2. تحويل التوقعات إلى أرقام الكلاسات الصحيحة (Argmax)
# استخدم argmax إذا كان لديك أكثر من كلاسين (Multi-class)
pred_labels = np.argmax(preds, axis=1)

# 3. جلب التسميات الحقيقية
true_labels = test_it.classes

# 4. طباعة النتائج
print("Confusion Matrix:")
print(confusion_matrix(true_labels, pred_labels))

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels))

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# 1. إعادة تجهيز بيانات الاختبار مع إيقاف الخلط العشوائي (shuffle=False) لضمان تطابق الترتيب
test_it_ordered = datagen.flow_from_dataframe(
    dataframe=test_df,
        directory=None,
            x_col='File_Path',
                y_col='Label',
                    target_size=(150, 150),  # تأكد أنها نفس الأبعاد المستخدمة في مشروعك
                        batch_size=32,
                            class_mode='categorical',
                                shuffle=False             # هذه هي النقطة الجوهرية لمنع الخطأ
                                )

                                # 2. حساب التوقعات بناءً على الترتيب الصحيح
                                preds = model.predict(test_it_ordered)
                                pred_labels = np.argmax(preds, axis=1)

                                # 3. جلب التسميات الحقيقية المرتبة
                                true_labels = test_it_ordered.classes

                                # 4. طباعة تقرير التصنيف النصي (Classification Report)
                                print("Classification Report:\n")
                                print(classification_report(true_labels, pred_labels, target_names=list(categories_dict.keys())))

                                # 5. حساب ورسم مصفوفة الارتباك بشكل مرئي واحترافي (Heatmap)
                                cm = confusion_matrix(true_labels, pred_labels)

                                plt.figure(figsize=(10, 8))
                                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                                            xticklabels=list(categories_dict.keys()),
                                                        yticklabels=list(categories_dict.keys()))
                                                        plt.ylabel('True Labels (القيم الحقيقية)')
                                                        plt.xlabel('Predicted Labels (توقعات النموذج)')
                                                        plt.title('Confusion Matrix Heatmap')
                                                        plt.show()
